# Runbook Grounding Evaluation\nQuick notebook to inspect retrieval and abstention behavior on benchmark questions.

In [ ]:
from pathlib import Path\nimport json\nfrom app.services.chunker import RunbookChunker\nfrom app.services.retrieval import RunbookRetrieval\nfrom app.services.answer_policy import AnswerPolicy

In [ ]:
runbook_dir = Path('../data/runbooks')\nbenchmark_path = Path('../data/benchmarks/benchmark_questions.json')\nchunks = RunbookChunker(runbook_dir).load_chunks()\nretrieval = RunbookRetrieval(chunks, mode='keyword')\npolicy = AnswerPolicy(min_confidence=0.35)\nbenchmarks = json.loads(benchmark_path.read_text())\nrows = []\nfor item in benchmarks:\n    top = retrieval.retrieve(item['question'], top_k=4)\n    response = policy.build_response(item['question'], top)\n    rows.append({\n        'question': item['question'],\n        'expected_runbook': item['expected_runbook'],\n        'insufficient': response.insufficient_evidence,\n        'confidence': response.confidence,\n        'top_runbook': top[0].runbook if top else None,\n    })\nrows[:10]